# B. Sanity Check

- **B1**: Initial loss ~ -log(1/C) (~0.693 for binary)
- **B2**: Overfit 20 samples (loss->0, acc->100%)
- **B3**: LR sweep to find sweet spot
- **B4**: Adam converges faster than SGD

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from network import Network, NetworkConfig
from optimizer import Adam, Sgd

## B1: Initial Loss ~ -log(1/C)

In [ ]:
np.random.seed(42)

config = NetworkConfig(
    layers=[30, 24, 24, 24, 2],
    activation='relu', loss='cross_entropy',
    output_activation='softmax', weights_initializer='heUniform'
)
net = Network(config)

x = np.random.randn(100, 30)
y = np.zeros((100, 2))
y[np.arange(100), np.random.randint(0, 2, 100)] = 1

output = net.forward(x)
initial_loss = net.loss(y, output)
expected = -np.log(1.0 / 2)

print(f'Initial loss:  {initial_loss:.4f}')
print(f'Expected:      {expected:.4f}')
print(f'Difference:    {abs(initial_loss - expected):.4f}')
print(f'Status: {"PASS" if abs(initial_loss - expected) < 0.3 else "FAIL"} (tolerance 0.3)')

## B2: Overfit 20 Samples

In [ ]:
np.random.seed(42)

config = NetworkConfig(
    layers=[30, 24, 24, 24, 2],
    activation='relu', loss='cross_entropy',
    output_activation='softmax', weights_initializer='heUniform'
)
net = Network(config)

n = 20
x = np.random.randn(n, 30)
y = np.zeros((n, 2))
y[np.arange(n), np.random.randint(0, 2, n)] = 1

optimizer = Adam(learning_rate=0.001)
losses = []

for epoch in range(500):
    out = net.forward(x)
    losses.append(net.loss(y, out))
    nw, nb = net.backward(y)
    optimizer.update(net, nw, nb)

final_out = net.forward(x)
final_loss = net.loss(y, final_out)
acc = np.mean(np.argmax(final_out, axis=1) == np.argmax(y, axis=1)) * 100

print(f'Final loss:     {final_loss:.6f}')
print(f'Final accuracy: {acc:.1f}%')
print(f'Status: {"PASS" if acc == 100.0 and final_loss < 0.01 else "FAIL"}')

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.title(f'Overfit {n} Samples (final acc: {acc:.0f}%)')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(True)
plt.show()

## B3: Learning Rate Sweep

In [ ]:
learning_rates = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
np.random.seed(42)
x = np.random.randn(50, 30)
y = np.zeros((50, 2))
y[np.arange(50), np.random.randint(0, 2, 50)] = 1

plt.figure(figsize=(10, 5))
results = {}

for lr in learning_rates:
    np.random.seed(42)
    config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
        loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
    net = Network(config)
    opt = Adam(learning_rate=lr)
    losses = []
    for _ in range(200):
        out = net.forward(x)
        loss = net.loss(y, out)
        if np.isnan(loss) or np.isinf(loss):
            losses.append(float('nan')); break
        losses.append(loss)
        nw, nb = net.backward(y)
        opt.update(net, nw, nb)
    results[lr] = losses[-1] if losses else float('nan')
    plt.plot(losses, label=f'lr={lr}')

plt.title('Learning Rate Sweep'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True); plt.yscale('log'); plt.show()

print('\nFinal losses:')
for lr, loss in results.items():
    print(f'  lr={lr:<8} -> {loss:.6f}')
best = min(results, key=results.get)
print(f'\nBest LR: {best}')

## B4: Adam vs SGD Convergence

In [ ]:
np.random.seed(42)
x = np.random.randn(50, 30)
y = np.zeros((50, 2))
y[np.arange(50), np.random.randint(0, 2, 50)] = 1

plt.figure(figsize=(10, 5))

for name, OptClass, lr in [('Adam (lr=0.001)', Adam, 0.001), ('SGD (lr=0.01)', Sgd, 0.01)]:
    np.random.seed(42)
    config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
        loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
    net = Network(config)
    opt = OptClass(learning_rate=lr)
    losses = []
    for _ in range(300):
        out = net.forward(x)
        losses.append(net.loss(y, out))
        nw, nb = net.backward(y)
        opt.update(net, nw, nb)
    print(f'{name}: final loss = {losses[-1]:.6f}')
    plt.plot(losses, label=name)

plt.title('Adam vs SGD'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True); plt.show()